# NB09: Evidence-Stratified Validation

**Purpose**: Stratify the Tier 1 vs RAST validation by UniProt evidence quality.

**Evidence dimensions**:
1. **Data source**: Swiss-Prot (575K reviewed) vs TrEMBL (214.6M unreviewed)
2. **Protein existence (PE)**: experimental (415K), transcript (1.5M), computational (homology + predicted, ~213M)

**Question**: Does evidence quality affect mapping accuracy? How much coverage comes from
high-evidence vs low-evidence annotations?

**Output**: Stratified F1 scores, coverage analysis, summary figure

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc

DATA_DIR = '../data'
FIG_DIR = '../figures'

## 1. Load Evidence Metadata

Swiss-Prot IDs were pre-extracted from `refdata_uniprot.entity`.
PE levels for experimental and transcript were pre-extracted from `refdata_uniprot.protein`.
Everything else in tier1 is computationally inferred (homology + predicted).

In [2]:
import sys
sys.path.insert(0, '../../scripts')
from berdl_notebook_utils.setup_spark_session import get_spark_session
import pyarrow as pa
import pyarrow.parquet as pq

spark = get_spark_session()

swissprot = set(pd.read_parquet(f'{DATA_DIR}/swissprot_proteins.parquet')['protein'])
print(f'Swiss-Prot proteins: {len(swissprot):,}')

pe_experimental = spark.sql("""
    SELECT REPLACE(protein_id, 'uniprot:', '') as protein
    FROM refdata_uniprot.protein
    WHERE evidence_for_existence = 'evidence at protein level'
""").toPandas()
pe_exp_set = set(pe_experimental['protein'])
print(f'PE experimental: {len(pe_exp_set):,}')
del pe_experimental

pe_transcript = spark.sql("""
    SELECT REPLACE(protein_id, 'uniprot:', '') as protein
    FROM refdata_uniprot.protein
    WHERE evidence_for_existence = 'evidence at transcript level'
""").toPandas()
pe_trans_set = set(pe_transcript['protein'])
print(f'PE transcript: {len(pe_trans_set):,}')
del pe_transcript

pe_uncertain = spark.sql("""
    SELECT REPLACE(protein_id, 'uniprot:', '') as protein
    FROM refdata_uniprot.protein
    WHERE evidence_for_existence = 'uncertain'
""").toPandas()
pe_unc_set = set(pe_uncertain['protein'])
print(f'PE uncertain: {len(pe_unc_set):,}')
del pe_uncertain

pe_counts = spark.sql("""
    SELECT evidence_for_existence, COUNT(*) as n
    FROM refdata_uniprot.protein
    GROUP BY evidence_for_existence
""").toPandas()
print(f'\nPE level totals (all UniProt):')
for _, row in pe_counts.iterrows():
    print(f'  {row["evidence_for_existence"]}: {row["n"]:,}')

gc.collect()

Swiss-Prot proteins: 574,627


PE experimental: 397,637


PE transcript: 1,384,293


PE uncertain: 1,731



PE level totals (all UniProt):
  inferred from homology: 83,813,057
  evidence at protein level: 414,976
  evidence at transcript level: 1,460,434
  predicted: 129,440,744
  uncertain: 1,731


47

## 2. Annotate Tier 1 Proteins

In [3]:
tier1 = pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet',
                        columns=['protein', 'ec'])
tier1 = tier1.drop_duplicates()
print(f'Tier 1 protein-EC pairs: {len(tier1):,}')

tier1['is_swissprot'] = tier1['protein'].isin(swissprot)

def assign_pe(protein):
    if protein in pe_exp_set:
        return 'experimental'
    elif protein in pe_trans_set:
        return 'transcript'
    elif protein in pe_unc_set:
        return 'uncertain'
    else:
        return 'computational'

unique_proteins = tier1[['protein']].drop_duplicates()
unique_proteins['pe_level'] = unique_proteins['protein'].map(assign_pe)
tier1 = tier1.merge(unique_proteins, on='protein', how='left')
del unique_proteins

print(f'\nData source distribution (Tier 1 protein-EC pairs):')
print(f'  Swiss-Prot: {tier1["is_swissprot"].sum():,}')
print(f'  TrEMBL:     {(~tier1["is_swissprot"]).sum():,}')

print(f'\nPE level distribution (Tier 1 protein-EC pairs):')
for level in ['experimental', 'transcript', 'computational', 'uncertain']:
    n = (tier1['pe_level'] == level).sum()
    print(f'  {level}: {n:,}')

print(f'\nUnique proteins per stratum:')
for col, vals in [('is_swissprot', [True, False]),
                  ('pe_level', ['experimental', 'transcript', 'computational'])]:
    for v in vals:
        n = tier1[tier1[col] == v]['protein'].nunique()
        print(f'  {col}={v}: {n:,}')

Tier 1 protein-EC pairs: 27,639,185



Data source distribution (Tier 1 protein-EC pairs):
  Swiss-Prot: 237,656
  TrEMBL:     27,401,529

PE level distribution (Tier 1 protein-EC pairs):


  experimental: 89,156
  transcript: 209,965


  computational: 27,339,954
  uncertain: 110

Unique proteins per stratum:
  is_swissprot=True: 218,318


  is_swissprot=False: 26,330,706
  pe_level=experimental: 79,078


  pe_level=transcript: 200,533


  pe_level=computational: 26,269,308


## 3. Stratified Protein-Level F1 (Swiss-Prot vs TrEMBL)

In [4]:
rast = pd.read_parquet(f'{DATA_DIR}/rast_protein_ec.parquet')
rast = rast.rename(columns={'protein_id': 'protein'})
rast_proteins = set(rast['protein'])

tier1_proteins = set(tier1['protein'])
shared_proteins = tier1_proteins & rast_proteins
del tier1_proteins

print(f'Shared proteins (Tier 1 ∩ RAST): {len(shared_proteins):,}')

rast_shared = rast[rast['protein'].isin(shared_proteins)][['protein', 'ec']].drop_duplicates()
del rast
gc.collect()
print(f'RAST pairs (shared): {len(rast_shared):,}')

Shared proteins (Tier 1 ∩ RAST): 15,952,112


RAST pairs (shared): 17,102,353


In [5]:
def compute_f1(pred_df, truth_df, label=''):
    merged = pred_df.merge(truth_df, on=['protein', 'ec'], how='outer', indicator=True)
    tp = int((merged['_merge'] == 'both').sum())
    fp = int((merged['_merge'] == 'left_only').sum())
    fn = int((merged['_merge'] == 'right_only').sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    del merged
    return {'stratum': label, 'tp': tp, 'fp': fp, 'fn': fn,
            'precision': prec, 'recall': rec, 'f1': f1,
            'pred_pairs': len(pred_df), 'truth_pairs': len(truth_df)}

results = []

# Overall (same as NB07)
t1_shared = tier1[tier1['protein'].isin(shared_proteins)][['protein', 'ec']].drop_duplicates()
results.append(compute_f1(t1_shared, rast_shared, 'ALL'))

# Swiss-Prot only
sp_shared = shared_proteins & swissprot
t1_sp = tier1[(tier1['protein'].isin(sp_shared)) & tier1['is_swissprot']][['protein', 'ec']].drop_duplicates()
rast_sp = rast_shared[rast_shared['protein'].isin(sp_shared)]
results.append(compute_f1(t1_sp, rast_sp, 'Swiss-Prot'))
del t1_sp, rast_sp

# TrEMBL only
tr_shared = shared_proteins - swissprot
t1_tr = tier1[(tier1['protein'].isin(tr_shared)) & ~tier1['is_swissprot']][['protein', 'ec']].drop_duplicates()
rast_tr = rast_shared[rast_shared['protein'].isin(tr_shared)]
results.append(compute_f1(t1_tr, rast_tr, 'TrEMBL'))
del t1_tr, rast_tr
gc.collect()

print(f'{"Stratum":<15s} {"Proteins":>10s} {"Pred":>10s} {"Truth":>10s} {"TP":>10s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 80)
for r in results:
    n_prot = len(sp_shared) if r['stratum'] == 'Swiss-Prot' else (len(tr_shared) if r['stratum'] == 'TrEMBL' else len(shared_proteins))
    print(f'{r["stratum"]:<15s} {n_prot:>10,} {r["pred_pairs"]:>10,} {r["truth_pairs"]:>10,} {r["tp"]:>10,} {r["precision"]:>6.1%} {r["recall"]:>6.1%} {r["f1"]:>6.3f}')

Stratum           Proteins       Pred      Truth         TP    Prec  Recall      F1
--------------------------------------------------------------------------------
ALL             15,952,112 16,700,160 17,102,353 14,149,439  84.7%  82.7%  0.837
Swiss-Prot         159,546    170,925    169,419    151,441  88.6%  89.4%  0.890
TrEMBL          15,792,566 16,529,235 16,932,934 13,997,998  84.7%  82.7%  0.837


## 4. Stratified Protein-Level F1 (PE Level)

In [6]:
pe_results = []
for level in ['experimental', 'transcript', 'computational']:
    level_proteins = set(tier1[tier1['pe_level'] == level]['protein']) & shared_proteins
    if not level_proteins:
        continue
    t1_pe = tier1[(tier1['protein'].isin(level_proteins)) & (tier1['pe_level'] == level)][['protein', 'ec']].drop_duplicates()
    rast_pe = rast_shared[rast_shared['protein'].isin(level_proteins)]
    r = compute_f1(t1_pe, rast_pe, f'PE: {level}')
    r['n_proteins'] = len(level_proteins)
    pe_results.append(r)
    del t1_pe, rast_pe
    gc.collect()

print(f'{"Stratum":<25s} {"Proteins":>10s} {"Pred":>10s} {"TP":>10s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 75)
for r in pe_results:
    print(f'{r["stratum"]:<25s} {r["n_proteins"]:>10,} {r["pred_pairs"]:>10,} {r["tp"]:>10,} {r["precision"]:>6.1%} {r["recall"]:>6.1%} {r["f1"]:>6.3f}')

Stratum                     Proteins       Pred         TP    Prec  Recall      F1
---------------------------------------------------------------------------
PE: experimental              29,410     33,203     24,757  74.6%  79.4%  0.769
PE: transcript                83,881     86,904     73,344  84.4%  85.2%  0.848
PE: computational         15,838,777 16,580,004 14,051,301  84.7%  82.7%  0.837


## 5. Reaction Coverage by Evidence Quality

In [7]:
ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
balanced_ids = set(
    pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', usecols=['id', 'status'])
    .query("status == 'OK'")['id']
    .str.replace('seed.reaction:', '', regex=False)
)

print(f'Reaction coverage by data source:')
for label, is_sp in [('Swiss-Prot', True), ('TrEMBL', False)]:
    ecs = set(tier1[tier1['is_swissprot'] == is_sp]['ec'])
    rxns = set(ec_bridge[ec_bridge['ec'].isin(ecs)]['rxn_bare'])
    print(f'  {label}: {len(ecs):,} ECs -> {len(rxns):,} reactions ({100*len(rxns)/len(balanced_ids):.1f}%)')

print(f'\nReaction coverage by PE level:')
for level in ['experimental', 'transcript', 'computational']:
    ecs = set(tier1[tier1['pe_level'] == level]['ec'])
    rxns = set(ec_bridge[ec_bridge['ec'].isin(ecs)]['rxn_bare'])
    print(f'  {level}: {len(ecs):,} ECs -> {len(rxns):,} reactions ({100*len(rxns)/len(balanced_ids):.1f}%)')

all_ecs = set(tier1['ec'])
all_rxns = set(ec_bridge[ec_bridge['ec'].isin(all_ecs)]['rxn_bare'])
print(f'\n  ALL Tier 1: {len(all_ecs):,} ECs -> {len(all_rxns):,} reactions ({100*len(all_rxns)/len(balanced_ids):.1f}%)')

sp_ecs = set(tier1[tier1['is_swissprot']]['ec'])
tr_only_ecs = set(tier1[~tier1['is_swissprot']]['ec']) - sp_ecs
sp_rxns = set(ec_bridge[ec_bridge['ec'].isin(sp_ecs)]['rxn_bare'])
tr_only_rxns = set(ec_bridge[ec_bridge['ec'].isin(tr_only_ecs)]['rxn_bare']) - sp_rxns
print(f'\nMarginal value of TrEMBL:')
print(f'  Swiss-Prot ECs: {len(sp_ecs):,} -> {len(sp_rxns):,} reactions')
print(f'  TrEMBL-exclusive ECs: {len(tr_only_ecs):,} -> {len(tr_only_rxns):,} additional reactions')

Reaction coverage by data source:


  Swiss-Prot: 4,509 ECs -> 15,172 reactions (44.2%)


  TrEMBL: 4,690 ECs -> 16,796 reactions (48.9%)

Reaction coverage by PE level:
  experimental: 4,571 ECs -> 15,894 reactions (46.3%)


  transcript: 2,436 ECs -> 12,567 reactions (36.6%)


  computational: 4,598 ECs -> 16,662 reactions (48.5%)

  ALL Tier 1: 5,032 ECs -> 17,215 reactions (50.1%)



Marginal value of TrEMBL:
  Swiss-Prot ECs: 4,509 -> 15,172 reactions
  TrEMBL-exclusive ECs: 523 -> 2,043 additional reactions


## 6. Cross-Tabulation: Swiss-Prot x PE Level

In [8]:
unique_prots = tier1[['protein', 'is_swissprot', 'pe_level']].drop_duplicates(subset=['protein'])
ct = pd.crosstab(unique_prots['pe_level'], unique_prots['is_swissprot'], margins=True)
ct.columns = ['TrEMBL', 'Swiss-Prot', 'Total']
print('Unique Tier 1 proteins by data source x PE level:')
print(ct.to_string())

Unique Tier 1 proteins by data source x PE level:
                 TrEMBL  Swiss-Prot     Total
pe_level                                     
computational  26100484      168824  26269308
experimental      42820       36258     79078
transcript       187402       13131    200533
uncertain             0         105       105
All            26330706      218318  26549024


## 7. Summary Figure

In [9]:
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
})

all_results = results + pe_results
labels = [r['stratum'] for r in all_results]
precs = [r['precision'] for r in all_results]
recs = [r['recall'] for r in all_results]
f1s = [r['f1'] for r in all_results]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels))
w = 0.25
ax.bar(x - w, precs, w, label='Precision', color='#1f77b4')
ax.bar(x, recs, w, label='Recall', color='#ff7f0e')
ax.bar(x + w, f1s, w, label='F1', color='#2ca02c')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('Protein-Level Validation by Evidence Quality (Tier 1 vs RAST)')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

ax.axvline(x=2.5, color='gray', linestyle='--', alpha=0.3)
ax.text(1, 1.02, 'By data source', ha='center', fontsize=9, color='gray', transform=ax.get_xaxis_transform())
ax.text(4, 1.02, 'By PE level', ha='center', fontsize=9, color='gray', transform=ax.get_xaxis_transform())

fig.savefig(f'{FIG_DIR}/evidence_stratified_f1.png')
plt.show()
print('Saved: evidence_stratified_f1.png')

Saved: evidence_stratified_f1.png


## 8. Summary

In [10]:
print('=' * 60)
print('NB09 EVIDENCE-STRATIFIED VALIDATION SUMMARY')
print('=' * 60)

print(f'\nBy data source:')
for r in results:
    print(f'  {r["stratum"]:<15s} F1={r["f1"]:.4f} (P={r["precision"]:.4f}, R={r["recall"]:.4f})')

print(f'\nBy PE level:')
for r in pe_results:
    print(f'  {r["stratum"]:<25s} F1={r["f1"]:.4f} (P={r["precision"]:.4f}, R={r["recall"]:.4f})')

print(f'\nKey takeaways:')
sp_f1 = next(r['f1'] for r in results if r['stratum'] == 'Swiss-Prot')
tr_f1 = next(r['f1'] for r in results if r['stratum'] == 'TrEMBL')
print(f'  Swiss-Prot vs TrEMBL F1 delta: {sp_f1 - tr_f1:+.4f}')

del tier1, t1_shared, rast_shared, ec_bridge
gc.collect()

NB09 EVIDENCE-STRATIFIED VALIDATION SUMMARY

By data source:
  ALL             F1=0.8372 (P=0.8473, R=0.8273)
  Swiss-Prot      F1=0.8899 (P=0.8860, R=0.8939)
  TrEMBL          F1=0.8366 (P=0.8469, R=0.8267)

By PE level:
  PE: experimental          F1=0.7690 (P=0.7456, R=0.7939)
  PE: transcript            F1=0.8482 (P=0.8440, R=0.8524)
  PE: computational         F1=0.8373 (P=0.8475, R=0.8273)

Key takeaways:
  Swiss-Prot vs TrEMBL F1 delta: +0.0533


69